# LLM Parallel Training and Inference

Эта глава посвящена тому, как обучают и запускают на инференс большие языковые модели, которые физически не помещаются ни в память, ни в вычислительный бюджет одного ускорителя. Цель — не дать рецепт настройки конкретного фреймворка, а показать логику развития идей: какая проблема возникала на каждом этапе роста моделей и какой приём её решал. Если держать в голове эту цепочку, то названия вроде FSDP, Megatron, ZeRO, vLLM перестают быть набором аббревиатур и складываются в понятную инженерную траекторию.

---

## Зачем вообще нужен параллелизм

Современная модель упирается одновременно в две стены: стену памяти и стену времени.

Стена памяти возникает потому, что обучение требует хранить не только сами веса. На каждый параметр приходится несколько связанных сущностей. Если обозначить число параметров через Ψ и считать в типичном режиме mixed precision с оптимизатором Adam, то на одно устройство ложится примерно: веса в половинной точности (2Ψ байт), градиенты в половинной точности (2Ψ байт) и состояния оптимизатора в полной точности — мастер-копия весов, первый и второй моменты Adam (вместе около 12Ψ байт). Итого порядка 16 байт на параметр ещё до учёта активаций. Для модели в 10 миллиардов параметров это уже около 160 ГБ только под состояние обучения, что превышает память любого отдельного ускорителя. Отдельно растут активации — промежуточные результаты прямого прохода, которые нужно хранить до обратного прохода, и их объём пропорционален размеру батча, длине последовательности и глубине сети.

Стена времени возникает потому, что обучение на терабайтах текста на одном устройстве заняло бы годы. Нужно задействовать сотни и тысячи ускорителей одновременно.

Параллелизм — это набор способов разложить модель, данные и вычисления по множеству устройств так, чтобы обойти обе стены. Дальше мы разберём эти способы примерно в том порядке, в котором они исторически появлялись, потому что каждый следующий обычно решал ограничение предыдущего.

## Язык коллективных коммуникаций

Прежде чем говорить о видах параллелизма, полезно зафиксировать словарь операций обмена данными между устройствами. Почти любой метод параллелизма — это, по сути, выбор того, какие из этих операций и когда выполнять.

All-reduce — каждое устройство имеет свой вектор, в конце на всех устройствах оказывается их сумма (или среднее). Это базовая операция синхронизации градиентов.

All-gather — каждое устройство имеет свой кусок данных, в конце на всех устройствах собран полный набор всех кусков.

Reduce-scatter — операция, обратная all-gather по структуре обмена: складываются вклады со всех устройств, но каждое устройство получает только свою часть результата.

All-to-all — каждое устройство отправляет каждому свою порцию данных и от каждого получает порцию; используется при маршрутизации в Mixture of Experts.

Broadcast — одно устройство рассылает данные всем остальным.

Важный факт, который многое объясняет: all-reduce алгоритмически равен композиции reduce-scatter и all-gather. Это наблюдение лежит в основе того, как из обычного data parallelism вырастает шардированный подход ZeRO.

---

## Data Parallelism

Это самый старый и самый простой подход, появившийся задолго до трансформеров. Идея: каждое устройство держит полную копию модели, но обрабатывает свой кусок батча. После прямого и обратного прохода устройства обмениваются градиентами через all-reduce, усредняют их и делают одинаковый шаг оптимизатора. Поскольку все начинали с одинаковых весов и применили одинаковый усреднённый градиент, копии остаются синхронными.

Data parallelism прекрасно масштабирует пропускную способность: вдвое больше устройств — примерно вдвое больший суммарный батч за то же время. Но у него есть фундаментальный недостаток для больших моделей — полная избыточность памяти. Каждое из N устройств хранит свою копию всех весов, всех градиентов и всех состояний оптимизатора. Если модель не помещается на одно устройство, то data parallelism сам по себе не помогает совсем: он не уменьшает требования к памяти, а лишь дублирует их.

Именно это противоречие — отличное масштабирование вычислений при полной избыточности памяти — задало главный вопрос следующего этапа: можно ли сохранить простоту data parallelism, но перестать хранить N копий одного и того же.

---

## ZeRO: устранение избыточности

ZeRO (Zero Redundancy Optimizer) из библиотеки DeepSpeed дал ключевой ответ. Наблюдение простое: в обычном data parallelism в каждый конкретный момент устройству нужны не все веса целиком, а лишь те, с которыми оно прямо сейчас считает. Состояния оптимизатора вообще нужны только в момент шага оптимизатора. Значит, можно не хранить всё на всех, а разбить (шардировать) состояние обучения по устройствам и подтягивать недостающее по требованию.

ZeRO вводит три нарастающие стадии шардирования.

Стадия 1 шардирует только состояния оптимизатора. Каждое устройство отвечает за обновление своей доли параметров и хранит моменты Adam лишь для неё. Это устраняет самую тяжёлую часть памяти при почти неизменной схеме коммуникаций.

Стадия 2 дополнительно шардирует градиенты. Вместо полного all-reduce, после которого у всех есть все градиенты, используется reduce-scatter: каждое устройство получает только агрегированные градиенты своей доли параметров — ровно те, которые ему нужны для обновления.

Стадия 3 шардирует и сами параметры. Теперь ни одно устройство не хранит полную модель. Перед тем как посчитать слой, устройства собирают его веса операцией all-gather, выполняют вычисление, а затем освобождают собранные веса. Память под параметры уменьшается линейно с числом устройств.

Концептуально ZeRO показал, что data parallelism и model parallelism — не взаимоисключающие лагеря. Можно сохранить программную модель data parallelism (каждое устройство видит свой батч и не нужно вручную разрезать слои), но при этом распределить хранение состояния как в model parallelism. Платой становится дополнительный сетевой обмен на сборку и роспуск весов, но this trade окупается возможностью обучать модели, которые иначе не поместились бы.

---

## Fully Sharded Data Parallel (FSDP)

FSDP — это нативная реализация идей ZeRO стадии 3 в PyTorch. Если ZeRO исторически был тесно связан с DeepSpeed, то FSDP сделал шардированный data parallelism стандартным гражданином экосистемы PyTorch, что сильно повлияло на его распространение.

Механика следующая. Модель разбивается на единицы (FSDP-юниты) — обычно это блоки трансформера, обёрнутые специальным образом. Параметры каждого юнита в покое лежат разрезанными по устройствам data-parallel группы. Когда подходит очередь юнита на прямом проходе, устройства делают all-gather и временно собирают его полные веса, выполняют вычисление и сразу же освобождают собранные веса, оставляя только свою долю. На обратном проходе веса снова собираются для вычисления градиентов, а полученные градиенты раздаются через reduce-scatter так, что каждому устройству достаётся агрегированный градиент только его доли параметров. Затем шаг оптимизатора делается локально над своей долей.

Важная инженерная деталь — перекрытие коммуникаций с вычислениями. Пока считается текущий юнит, в фоне уже идёт all-gather следующего (prefetch). При удачном перекрытии накладные расходы на сеть прячутся за полезными вычислениями, и FSDP по скорости приближается к обычному data parallelism, но без его избыточности по памяти.

Хорошая ментальная модель: FSDP — это data parallelism, который в каждый момент материализует ровно тот кусок модели, который нужен прямо сейчас, и больше ничего.

---

## Tensor Parallelism

Параллелизм по данным и его шардированные варианты разбивают хранение, но каждое отдельное матричное умножение по-прежнему происходит целиком на одном устройстве (пусть и после сборки весов). Tensor parallelism идёт глубже и разрезает сами матрицы внутри одного слоя, распределяя одно умножение между несколькими устройствами. Это внутрислойный, или intra-layer, параллелизм. Канонической реализацией стал Megatron-LM.

Рассмотрим блок из двух линейных слоёв в feed-forward части трансформера. Первую матрицу разрезают по столбцам: каждое устройство держит свой набор столбцов и вычисляет свою часть скрытого представления независимо. Между линейными слоями стоит нелинейность (например, GeLU), которая применяется поэлементно, поэтому её можно посчитать на каждом куске локально без обмена. Вторую матрицу разрезают по строкам так, чтобы частичные результаты от всех устройств в сумме давали правильный выход; эта сумма собирается операцией all-reduce. В блоке внимания аналогично распределяют головы внимания: каждое устройство считает свой набор голов целиком.

Tensor parallelism позволяет уместить очень широкий слой, который не влезает на одно устройство, и сократить латентность одного шага, поскольку умножение идёт параллельно. Расплата — частые операции all-reduce внутри каждого слоя, очень чувствительные к пропускной способности сети. Поэтому tensor parallelism практически всегда держат внутри одного узла, где устройства соединены быстрым межсоединением (например, NVLink), и редко растягивают между узлами.

---

## Pipeline Parallelism

Если tensor parallelism режет слои поперёк (внутри слоя), то pipeline parallelism режет модель вдоль — по слоям. Это межслойный, inter-layer параллелизм. Модель разбивается на последовательные стадии: первые несколько слоёв живут на устройстве 1, следующие — на устройстве 2 и так далее. Активации передаются от стадии к стадии как по конвейеру.

Наивная реализация неэффективна: пока первое устройство считает первый слой, остальные простаивают, ожидая своей очереди. Прорыв принесла идея микробатчей (GPipe): большой батч делится на много мелких микробатчей, которые запускают в конвейер один за другим. Как только первая стадия закончила первый микробатч и передала его дальше, она тут же берётся за второй. В установившемся режиме все стадии заняты одновременно.

Тем не менее остаётся неустранимый эффект пузыря (pipeline bubble) — простой в начале наполнения конвейера и в конце его опустошения. Доля простоя тем меньше, чем больше микробатчей приходится на число стадий. Дальнейшие работы боролись именно с пузырём и с памятью: PipeDream предложил расписание 1F1B (one-forward-one-backward), при котором прямые и обратные проходы чередуются так, чтобы раньше освобождать память от активаций; Megatron добавил перемежающееся (interleaved) расписание, где каждое устройство держит не одну непрерывную стадию, а несколько разнесённых, что ещё сильнее сжимает пузырь.

Сильная сторона pipeline parallelism — скромные требования к сети: между стадиями передаются только активации на их границах, и делать это можно относительно редко, поэтому стадии нормально разносить между узлами. Слабая сторона — пузырь и более сложная организация расписания.

---

## Sequence и Context Parallelism

С ростом длины контекста узким местом стали активации, объём которых растёт с длиной последовательности, и память под механизм внимания, чья стоимость квадратична по длине. Появилось семейство приёмов, режущих вычисление вдоль оси последовательности.

Sequence parallelism в трактовке Megatron дополняет tensor parallelism: те части блока, которые tensor parallelism не разрезает (например, нормализация и dropout, работающие поэлементно), разбиваются вдоль размерности последовательности, что снимает дублирование активаций по этим участкам.

Context parallelism идёт дальше и распределяет саму длинную последовательность между устройствами для вычисления внимания. Ярким представителем является Ring Attention: устройства держат свои отрезки последовательности и пересылают блоки ключей и значений по кольцу, постепенно накапливая результат внимания так, что никогда не требуется материализовать полную матрицу внимания целиком на одном устройстве. Это открывает дорогу к контекстам в сотни тысяч и миллионы токенов.

## Expert Parallelism и Mixture of Experts

Все предыдущие приёмы исходили из того, что для каждого токена прогоняется вся модель. Mixture of Experts (MoE) меняет саму архитектуру ради экономии вычислений. Вместо одного feed-forward блока в слое размещают много параллельных блоков-экспертов и добавляют маршрутизатор (router), который для каждого токена выбирает лишь небольшое подмножество экспертов (например, одного-двух из десятков). В результате суммарное число параметров модели огромно, а вычислений на токен — как у небольшой плотной модели. Это разреженная активация: используется лишь малая доля весов на каждый токен.

Ключевые вехи — GShard, показавший масштабирование MoE до триллионов параметров, и Switch Transformer, упростивший маршрутизацию до выбора одного эксперта на токен.

С точки зрения распределённых вычислений MoE порождает свой вид параллелизма — expert parallelism: эксперты раскладываются по разным устройствам. Поскольку маршрутизатор может отправить токены любого устройства к экспертам на любом другом устройстве, требуется операция all-to-all — сначала чтобы разослать токены к их экспертам, потом чтобы собрать результаты обратно. Отсюда же главные сложности MoE: балансировка нагрузки между экспертами (чтобы какой-то эксперт не оказался перегружен, а другой — простаивал) и высокая чувствительность к качеству и стоимости all-to-all коммуникаций.

## 3D Parallelism: комбинирование подходов

Перечисленные виды параллелизма не конкурируют, а складываются в слои, и обучение моделей предельного масштаба использует их одновременно. Классическая комбинация — так называемый 3D parallelism, сочетающий tensor, pipeline и data parallelism (а при наличии MoE добавляется и expert parallelism).

Распределяют их с учётом стоимости коммуникаций. Tensor parallelism, как самый требовательный к сети, держат внутри узла на быстром межсоединении. Pipeline parallelism, передающий данные редко и небольшими порциями, растягивают между узлами. Data parallelism (часто в шардированной форме) делают внешним слоем, реплицируя всю эту конструкцию для масштабирования по данным. Такая компоновка и позволяет занять тысячи ускорителей при обучении моделей масштаба десятков и сотен миллиардов параметров.

## Техники экономии памяти

Параллелизм работает в связке с приёмами, которые уменьшают потребление памяти на каждом отдельном устройстве. Они часто не менее важны, чем сам выбор схемы параллелизма.

Activation checkpointing (он же gradient checkpointing, или recomputation) — это размен вычислений на память. Вместо того чтобы хранить все активации прямого прохода до обратного, сохраняют лишь немногие контрольные точки, а недостающие активации пересчитывают заново во время обратного прохода. Память под активации резко падает ценой повторного прямого прохода для части сети.

Mixed precision training выполняет вычисления в половинной точности (fp16 или bf16), сохраняя при этом мастер-копию весов в полной точности fp32 для устойчивости обновлений. Это уменьшает память и ускоряет вычисления на тензорных ядрах. Формат bf16 со временем стал предпочтительным из-за большего динамического диапазона и меньших проблем с переполнением.

Offloading выгружает то, что не нужно прямо сейчас, на более медленную, но более ёмкую память. ZeRO-Offload переносит состояния оптимизатора и градиенты в оперативную память хоста, а ZeRO-Infinity идёт дальше и задействует NVMe-накопители, позволяя обучать очень большие модели на ограниченном числе ускорителей ценой пропускной способности.

## Параллелизм на инференсе

Инференс устроен иначе, чем обучение, поэтому и приёмы распределения другие. Главное отличие — отсутствие обратного прохода и состояний оптимизатора, зато появляется авторегрессионная генерация токен за токеном и связанный с ней кэш.

Инференс распадается на две фазы с разным характером. Фаза префилла (prefill) обрабатывает весь входной промпт сразу и упирается в вычисления — это режим, ограниченный вычислительной мощностью. Фаза декодирования (decode) генерирует по одному токену за шаг и упирается в память и пропускную способность, потому что на каждый новый токен нужно прочитать все веса модели ради одного маленького умножения — это режим, ограниченный памятью.

Чтобы не пересчитывать внимание по всей истории на каждом шаге, хранят KV-cache — кэш ключей и значений для уже сгенерированных токенов. Этот кэш растёт линейно с длиной последовательности и с числом одновременных запросов и быстро становится главным потребителем памяти на инференсе.

Для распределения самой модели на инференсе используют те же tensor и pipeline parallelism. Tensor parallelism применяют, чтобы крупная модель поместилась на нескольких устройствах и чтобы снизить латентность одного шага генерации. Pipeline parallelism помогает поднять пропускную способность при обслуживании многих запросов. Шардированный data parallelism в стиле FSDP на инференсе обычно не нужен, так как нет состояний оптимизатора и градиентов.

Поверх этого выросли приёмы, специфичные именно для обслуживания LLM.

PagedAttention, предложенный в системе vLLM, управляет KV-кэшем подобно виртуальной памяти операционной системы: кэш хранится не одним непрерывным куском на запрос, а страницами (блоками), что почти устраняет фрагментацию и позволяет держать в памяти больше одновременных запросов.

Continuous batching (идея системы Orca) меняет гранулярность батчинга: вместо того чтобы ждать, пока завершатся все запросы в батче, система добавляет новые и убирает завершившиеся запросы на уровне отдельных шагов генерации. Это держит ускорители занятыми и резко повышает пропускную способность при разной длине ответов.

Speculative decoding ускоряет генерацию, нападая на её последовательную природу. Маленькая быстрая черновая модель предлагает сразу несколько следующих токенов, а большая модель за один проход проверяет их параллельно и принимает совпавший префикс. Если черновая модель угадывает хорошо, за один тяжёлый проход подтверждается несколько токенов сразу.

Квантизация (понижение точности весов и иногда активаций до INT8, INT4 и подобных форматов) уменьшает объём памяти под веса и под KV-кэш и ускоряет чтение из памяти — что особенно ценно в ограниченной памятью фазе декодирования.

## Как развивалась мысль: краткая траектория

Полезно собрать всё в одну линию рассуждения, потому что именно эта логика хорошо ложится в рассказ и хорошо звучит на собеседовании.

Сначала был простой data parallelism: масштабируем вычисления, дублируя модель. Он уперся в избыточность памяти, как только модели перестали помещаться на одно устройство.

В ответ появились два независимых направления. Со стороны архитектуры модели — model parallelism: tensor parallelism начал резать слои внутри (Megatron), pipeline parallelism — между слоями (GPipe, PipeDream). Со стороны памяти — ZeRO предложил шардировать состояние обучения, сохранив удобную программную модель data parallelism; эта идея закрепилась в индустрии как FSDP.

Дальше масштаб контекста и масштаб параметров потребовали новых осей разрезания: sequence и context parallelism (вплоть до Ring Attention) для длинных последовательностей, expert parallelism для разреженных MoE-моделей. Всё это научились сочетать в 3D parallelism, раскладывая виды параллелизма по уровням сети согласно их аппетиту к коммуникациям, и подпирать приёмами экономии памяти — recomputation, mixed precision, offloading.

Наконец, когда обученные модели понадобилось дёшево и быстро обслуживать, фокус сместился на инференс, у которого своя физика: авторегрессия, две фазы с разным узким местом и доминирующий KV-кэш. Отсюда выросли PagedAttention, continuous batching, speculative decoding и квантизация.

Сквозная мысль всей главы одна: каждый новый приём параллелизма — это ответ на конкретное ставшее доминирующим ограничение (память на параметры, память на активации, латентность шага, стоимость коммуникаций, авторегрессионная последовательность инференса). Если запоминать не аббревиатуры, а пары проблема-решение, вся область укладывается в стройную и легко воспроизводимую картину.